<a href="https://colab.research.google.com/github/alwjr-hccs/ITAI_ML_FirstRepo_AWilliams/blob/main/Phishing_Analysis_Using_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
bash


Copy
!curl -fsSL https://ollama.com/install.sh | sh

NameError: name 'bash' is not defined

In [2]:
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama, daemon=True)
thread.start()
time.sleep(3)  # Give the server a moment to initialize
print("Ollama server started.")

Exception in thread Thread-3 (run_ollama):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_545/2804685223.py", line 6, in run_ollama
  File "/usr/lib/python3.12/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.12/subprocess.py", line 1955, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'ollama'


Ollama server started.


In [3]:
import requests

response = requests.get("http://localhost:11434")
print(response.text)  # Should print: "Ollama is running"

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7ee639ed3ce0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [4]:
!ollama pull gemma2:9b-instruct-q4_K_M

/bin/bash: line 1: ollama: command not found


In [5]:
!ollama pull phi3:mini          # ~2.3 GB, very fast
!ollama pull mistral:7b-q4_K_M # ~4 GB, strong reasoning

/bin/bash: line 1: ollama: command not found
/bin/bash: line 1: ollama: command not found


In [6]:
!pip install beautifulsoup4 chardet instructor pydantic openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 16.5 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.15.0
    Uninstalling jiter-0.15.0:
      Successfully uninstalled jiter-0.15.0


In [7]:
from bs4 import BeautifulSoup
import chardet

def clean_email(raw: str) -> str:
    # Detect encoding
    detected = chardet.detect(raw.encode())
    encoding = detected.get("encoding", "utf-8") or "utf-8"

    # Strip HTML
    soup = BeautifulSoup(raw, "html.parser")
    text = soup.get_text(separator="\n")

    # Normalize whitespace
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return "\n".join(lines)


In [11]:
from pydantic import BaseModel, Field
from typing import List, Literal

class PhishingAnalysis(BaseModel):
    classification: Literal["phishing", "suspicious", "benign"]
    confidence: float = Field(ge=0.0, le=1.0, description="0.0 = uncertain, 1.0 = certain")
    iocs: List[str] = Field(description="Extracted URLs, domains, IPs, sender addresses")
    reasons: List[str] = Field(description="Bullet-point rationale for the classification")
    recommended_actions: List[str] = Field(
        description="SOC next steps: quarantine, block, search mailbox, etc."
    )

In [13]:
import instructor
from openai import OpenAI

# Point the OpenAI client at the local Ollama server
client = instructor.from_openai(
    OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"),
    mode=instructor.Mode.JSON,
)

In [15]:
def analyze_email(email_text: str) -> PhishingAnalysis:
    cleaned = clean_email(email_text)
    return client.chat.completions.create(
        model="gemma2:9b-instruct-q4_K_M",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a SOC Tier-1 analyst. Analyze the email below for phishing indicators. "
                    "Be precise. Only flag indicators that are present in the text."
                )
            },
            {"role": "user", "content": cleaned},
        ],
        response_model=PhishingAnalysis,
        max_retries=3,
    )


In [16]:
import ipywidgets as widgets
from IPython.display import display, clear_output

email_box = widgets.Textarea(placeholder="Paste email here...", layout=widgets.Layout(width="100%", height="200px"))
run_button = widgets.Button(description="Analyze", button_style="danger")
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        result = analyze_email(email_box.value)
        print(f"Classification : {result.classification.upper()}")
        print(f"Confidence     : {result.confidence:.0%}")
        print(f"IOCs           : {', '.join(result.iocs) or 'None detected'}")
        print("\nReasons:")
        for r in result.reasons: print(f"  • {r}")
        print("\nRecommended Actions:")
        for a in result.recommended_actions: print(f"  → {a}")

run_button.on_click(on_click)
display(email_box, run_button, output)

Textarea(value='', layout=Layout(height='200px', width='100%'), placeholder='Paste email here...')

Button(button_style='danger', description='Analyze', style=ButtonStyle())

Output()

In [17]:
import pandas as pd
from tqdm.notebook import tqdm

# Load your dataset (CSV with 'email_text' and 'true_label' columns)
df = pd.read_csv("phishing_dataset.csv")
results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    try:
        r = analyze_email(row["email_text"])
        results.append({
            "true_label": row["true_label"],
            "predicted": r.classification,
            "confidence": r.confidence,
            "iocs": r.iocs,
        })
    except Exception as e:
        results.append({"true_label": row["true_label"], "predicted": "error", "error": str(e)})

results_df = pd.DataFrame(results)

FileNotFoundError: [Errno 2] No such file or directory: 'phishing_dataset.csv'

In [18]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print(classification_report(results_df["true_label"], results_df["predicted"]))

cm = confusion_matrix(results_df["true_label"], results_df["predicted"])
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["benign","phishing","suspicious"],
            yticklabels=["benign","phishing","suspicious"])
plt.title("Phishing Classifier Confusion Matrix")
plt.show()

NameError: name 'results_df' is not defined